# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Additional Notes**
- Using sim_year as the actual year 

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

---
**!!! ToDo**
- make flow diagram of model
- parallelize the code
- CONTINUE add Confidence Intervals for Return Levels

# Import Libraries

In [ ]:
import sys
import random
import time
from datetime import datetime
from glob import glob
import warnings

import xarray as xr
from IPython.display import Markdown, display
from pandas import DataFrame

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)

# Settings

In [ ]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [ ]:
hindcast_start = 1960
hindcast_end = 2026

In [ ]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [ ]:
dic_timing = {}
dic_notes_analysis = {}

In [ ]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [ ]:
print_msg = True

export_report=True
display_results = False
save_regression_summary = True

# Import data

In [ ]:
dic_timing['import data'] = {}
dic_timing['import data']['start'] = datetime.now()

ls_files = [file for file in glob(path + '*.nc')]
ls_files

In [ ]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

dic_timing['import data']['end'] = datetime.now()

# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, using `joblib` - saving ~60% (from 3min30sec down to 1min22sec)


In [ ]:
dic_timing['data correction'] = {}
dic_timing['data correction']['start'] = datetime.now()
dic_data_per_model = dbf.data_preparation(ls_files=ls_files, dic_data_per_model=dic_data_per_model)
dic_timing['data correction']['end'] = datetime.now()

In [ ]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))
dic_notes_analysis['data overview'] = ut.create_data_overview(dic_data_per_model,ls_files)

display(Markdown(f"Execution time · {dic_timing['data correction']['end'] - dic_timing['data correction']['start']}sec"))

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)


In [ ]:
dic_timing['data pooling'] = {}
dic_timing['data pooling']['start'] = datetime.now()
ls_notes = []

# ------------------------------------------------------------------------------------------
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")
dic_timing['data pooling']['end'] = datetime.now()

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
message = f"""
        Overview of combined dataset
        \tFinal dimensions: {combined.dims}
        \tShape: {combined.shape}
        \tNumber of models: {combined.model.size}
        \tNumber of locations: {combined.location.size}
        """.strip()
print(message)
ls_notes.append(message)

display(Markdown(f"\n**Create and Store summary of locations for which we have no data in either of the models**"))
missing_locations = dbf.create_summary_location_w_missing_data(
    dic_data_per_model=dic_data_per_model, combined=combined, 
    dir_export='/'.join(path_export.split('/')[:-3]) + '/exploration'
    )
ls_notes.append(f'{len(missing_locations)} locations without any valid data found!')

# ------------------------------------------------------------------------------------------
dic_notes_analysis['data pooling'] = ls_notes

### Validation Check

In [ ]:
dic_timing['data validity check'] = {}
dic_timing['data validity check']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
list_model_labels = list(dic_data_per_model.keys())

model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

In [ ]:
if (
    lon_target == lon_rev
    and lat_target == lat_rev
    and data_for_model_for_location.dropna().equals(revised_dataset.dropna()
    )
):
    message = "Validity Check performed successfully"
else:
    message = "Validity Check failed"

dic_timing['data validity check']['end'] = datetime.now()
dic_notes_analysis['data pooling'].append(message + ' ' + str(dic_timing['data validity check']['end']))

# Workflow GEV - Generalized Extreme Value

## OPTION1
Using all data available per location - from all years and models. Currently, selecting a subset of 10 sites, but later will be applied to all locations.

#### Initial trial with subset of 10 locations

In [ ]:
dic_timing['GEV approach 1'] = {}
dic_timing['GEV approach 1']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
list_sites = random.choices(range(combined.shape[-1]),k=10)

dic_data_per_location = {}
for loc_ex in list_sites: 
    data_at_location = combined[:,:,:, loc_ex].to_dataframe().dropna().reset_index()
    data_at_location = data_at_location.rename(columns={'annualMax':'storm_surge'})
    dic_data_per_location[loc_ex] = data_at_location

### Run Analysis

Note, the output is stored as 
- visuals → png
- tabular data (DataFrames) → Parquet
- other objects (dicts, strings, floats) → Pickle

File structure
```results/
├─ location_1/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
├─ location_2/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
...
```

In [ ]:
orig_stdout, orig_stderr, fh, logger, log_path = ut.initialize_logger(
    f"LOGS_GEVAnalysis_pooled_{datetime.now():%Y%m%d_%H%M%S}.log"
    )

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['analysis start'] = datetime.now()
ls_notes = []

print("\n" + "="*100)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER LOCATION")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_prepared, notes = ut.prepare_pooled_data(
    dic_data=dic_data_per_location,
    hindcast_start=hindcast_start,
    hindcast_end=hindcast_end
)
ls_notes.append(notes)

# ------------------------------------------------------------------------------------------
results = {}
for loc_id, df_prepared in dic_prepared.items():
    print("\n" + "-"*70)
    print(f"Analyzing location id {loc_id} ...")

    lon_loc = df_prepared.lon.unique()[0]
    lat_loc = df_prepared.lat.unique()[0]

    print("\tLookup location info...")
    location_closest = dbf.add_location_labels(DataFrame([lon_loc, lat_loc], index=['lon', 'lat']).T)
    location_info = ' '.join(location_closest.values[0][2:])
    print(f"\t → Closest location identified: {location_info}")

    result, ls_warnings = gev.analyze_per_location(
        df_prepared, loc_id, lat_loc, lon_loc, location_info, return_periods
    )
    if any(ls_warnings):
        ls_notes.append(ls_warnings)
    
    if result is None:
        message = f"\t→ Warning! No valid GEV fit for location id {loc_id}. Skipping ..."
        print(message)
        ls_notes.append(message)
        continue
    
    if export_report and path_export: 
        export_path_site = ut.save_location_results(            
            location_id=loc_id, result_location=result, base_dir=path_export, 
            plot_period_evolution=plot_period_evolution, display_results=display_results
            )

    result['file_path_report'] = export_path_site
    results[loc_id] = result

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['analysis end'] = datetime.now()

print("\n" + "="*100)
time_diff = dic_timing['GEV approach 1']['analysis end'] - dic_timing['GEV approach 1']['analysis start']
print(f"✓ ANALYSIS COMPLETED IN {time_diff}!")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_notes_analysis['GEV pooled analysis'] = ls_notes
sys.stdout = orig_stdout
sys.stderr = orig_stderr
logger.removeHandler(fh)
fh.close()

10 locations require an execution time of ~ 30-50sec<br> 
Upscaling to 11022 locations, will result in an execution time of ~9-10hours!

## OPTION2
Re-run stationary GEV per year (for location parameter; scale and shape remain as globally defined)

In [ ]:
# del results
try:
    results.keys()
    print('✓ continue with available dictionary')
    
except NameError:
    print('import data from files...')
    results = ut.import_results_from_files(path_export)

In [ ]:
dic_timing['GEV approach 2'] = {}
dic_timing['GEV approach 2']['analysis start'] = datetime.now()

# ------------------------------------------------------------------------------------------
results_extended, ls_notes_analysis = gev.execute_and_store_stat_gev_per_year(
    results=results, store_results=False, return_periods=return_periods
    )

# ------------------------------------------------------------------------------------------
for key, outer_list in ls_notes_analysis.items():
    ls_notes_analysis[key] = [inner for inner in outer_list if inner]
dic_notes_analysis['annual_statGEV'] = ls_notes_analysis
dic_timing['GEV approach 2']['analysis end'] = datetime.now()

# ------------------------------------------------------------------------------------------
time_diff = dic_timing['GEV approach 2']['analysis end'] - dic_timing['GEV approach 2']['analysis start']
print(f"Execution time for computing GEV per year: {time_diff}sec")

---
test including CI for annual stationary GEV

In [ ]:
import os 



In [ ]:
results_extended, ls_notes_analysis = execute_and_store_stat_gev_per_year(results, store_results=False, return_periods=return_periods)

Upscaling to 11022 locations, will result in an execution time of ~12.5hours!

### Regression of location parameter over years
including uncertainty given by n_obs

**NOTE**<br>
> centering the year parameter due to the following warning:<br>
*"The condition number is large, 2.37e+05. This might indicate that there are strong multicollinearity or other numerical problems."*

In [ ]:
dic_timing['GEV approach 2']['regression start'] = datetime.now()

# ------------------------------------------------------------------------------------------
en = 0
for site_id, dic_location in results_extended.items():
    en+=1
    print(
        f"\nPlotting GEV μ trend analysis for location id {site_id} (#{en}/{len(results_extended.keys())})...", 
        end="\r"
        )
    
    df_stat_gev_per_year = dic_location['fit results']['gev_stationary']['analysis_per_year'].dropna()
    df = df_stat_gev_per_year.reset_index().rename(columns={'index': 'year'})

    global_statgev_shape = dic_location['fit results']['gev_stationary']['shape']
    global_statgev_scale = dic_location['fit results']['gev_stationary']['scale']

    [
        df, wls_delta, weights, y_pred, year_grid, year_mean
        ] = gev.weighted_least_square_regression_annual_location(global_statgev_scale, global_statgev_shape, df)
    results_extended[site_id]['WLSdelta'] = dict({'summary': wls_delta, 'weights': weights})
    
    fig = dbplt.plot_gev_mu_trend(
        df=df,
        weights=weights,
        year_grid=year_grid,
        year_mean=year_mean,
        y_pred=y_pred,
        wls_delta=wls_delta,
        nonstat_years=dic_location['data'].year.values.astype(int), 
        nonstat_ci=dic_location['fit results']['gev_nonstationary']['CI'],
        display_results=display_results,
        colors_reg=['#333333FF', '#C88D35FF'],
        markers_color='#99E3DDFF'
        )
    
    if save_regression_summary and ('file location' in dic_location.keys() or 'file_path_report' in dic_location.keys()):
        if 'file location' in dic_location.keys():
            save_path = dic_location['file location']
        else:
            save_path = dic_location['file_path_report']
        with open(save_path + '/WLSdelta_summary.html', 'w') as f:
            f.write( wls_delta.summary().as_html())
        
        lat = str(dic_location['location info']['lat'].round(3))
        lon = str(dic_location['location info']['lon'].round(3))
        country = dic_location['location info']['description'].split(',')[-1].strip()  
        file_name = f"/GEVTrendAnalysis_location_{str(site_id)}_{country}_{lat}|{lon}.png"
        fig.savefig(save_path+file_name, dpi=300, bbox_inches='tight')

    else:
        print("\t skipping saving GEV μ trend analysis ...")

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 2']['regression end'] = datetime.now()
time_diff = dic_timing['GEV approach 2']['regression end'] - dic_timing['GEV approach 2']['regression start']
print(
    f"\nExecution time for computing regression analysis for annual stationary GEV and non-stationary GEV: "
    f"{time_diff}sec"
    )

# Create high-level Summary

##  Store Run Notes and Warnings

In [ ]:
ut.store_analysis_notes(dic_notes_analysis, path_export)

# Parameters Summary

In [ ]:
site_id = 2050

In [ ]:
DataFrame([
    results[site_id]['location info']['lat'], 
    results[site_id]['location info']['lon'], 
    results[site_id]['location info']['description']
], index=['lat', 'lon', 'closest point identified']).T

In [ ]:
DataFrame([
    results[site_id]['hindcast period'][0], results[site_id]['hindcast period'][1]
    ], index=['start', 'end'], columns=['hindcast period']).T

In [ ]:
results[site_id]['fit results']['gev_nonstationary']

In [ ]:
results[site_id]['fit results']['gev_stationary']

In [ ]:
results[site_id]['model_comparison']

In [ ]:
from pandas import concat

In [ ]:
wls_params = concat([results[site_id]['WLSdelta']['summary'].params, results[site_id]['WLSdelta']['summary'].bse], axis=1)
wls_params.columns = ['value', 'std err']
wls_params

In [ ]:
results[site_id]['WLSdelta']['summary'].summary()

In [ ]:
results[site_id]['return_levels']

# Next Steps

summarize the intercept, slopes, CIs for different approaches and store results in human-readable format

#### Potential Visualizations
- Maps of 100-year return levels along European coastline
- Difference maps: non-stationary minus stationary → climate change impact
- Probability exceedance curves for selected cities
- Histograms / density of return levels → compare regions
- Time series of non-stationary μ or return levels → show increasing trends

In [ ]:
site_id = 5943
result_location = results_extended[site_id]

result_location.keys()

In [ ]:
result_location['location info']

In [ ]:
result_location['hindcast period'] # years

In [ ]:
# GEV Stationary
result_location['fit results']['gev_stationary']['analysis_per_year']

In [ ]:
# GEV non-Stationary
result_location['fit results']['gev_nonstationary']

In [ ]:
result_location['model_comparison']

In [ ]:
result_location['return_levels']